In [1]:
import pandas as pd
import numpy as np


class Baseline:
    def __init__(
        self,
        lista_produtos: list[str],
        df_contratos: pd.DataFrame,
        df_interacoes: pd.DataFrame,
        df_clientes: pd.DataFrame,
    ):
        self.lista_produtos = lista_produtos
        self.df_contratos = df_contratos
        self.df_interacoes = df_interacoes
        self.df_clientes = df_clientes

        self.df_ranking_geral = (
            self.df_interacoes.groupby("produto")["contratou"]
            .sum()
            .sort_values(ascending=False)
        )

        self.df_ranking_segmento = (
            pd.merge(
                self.df_interacoes,
                self.df_clientes[["id_cliente", "segmento"]],
                on="id_cliente",
            )
            .groupby(["segmento", "produto"])[["contratou"]]
            .sum()
            .reset_index()
            .sort_values(by=["segmento", "contratou"], ascending=False)
        )

    def get_elegiveis_cliente(self, id_cliente: str, timestamp: str) -> list[str]:

        ativos = set(
            self.df_contratos[
                (self.df_contratos["id_cliente"] == id_cliente)
                & (self.df_contratos["status"] == "ativo")
                & (self.df_contratos["data_contratacao"] < timestamp[:10])
            ]["produto"]
        )

        return [p for p in self.lista_produtos if p not in ativos]

    def gerar_baseline_aleatorio(self, id_cliente: str, timestamp: str):

        elegiveis = self.get_elegiveis_cliente(id_cliente, timestamp)

        np.random.shuffle(elegiveis)
        scores = np.random.rand(len(elegiveis))

        ranking = (
            pd.DataFrame({"produto": elegiveis, "score": scores})
            .sort_values(by="score", ascending=False)
            .reset_index(drop=True)
        )

        ranking["posicao"] = ranking.index + 1
        return ranking

    def gerar_baseline_popularidade(self, id_cliente, timestamp: str):

        produtos_elegiveis = self.get_elegiveis_cliente(id_cliente, timestamp)
        scores = self.df_ranking_geral[produtos_elegiveis].values

        ranking = (
            pd.DataFrame({"produto": produtos_elegiveis, "score": scores})
            .sort_values(by="score", ascending=False)
            .reset_index(drop=True)
        )

        ranking["posicao"] = ranking.index + 1
        return ranking

    def gerar_baseline_popularidade_segmento(self, id_cliente, timestamp: str):

        segmento = self.df_clientes[self.df_clientes["id_cliente"] == id_cliente][
            "segmento"
        ].iloc[0]

        produtos_elegiveis = self.get_elegiveis_cliente(id_cliente, timestamp)

        ranking = (
            self.df_ranking_segmento[
                (self.df_ranking_segmento["segmento"] == segmento)
                & (self.df_ranking_segmento["produto"].isin(produtos_elegiveis))
            ][["produto", "contratou"]]
            .rename(columns={"contratou": "score"})
            .sort_values(by="score", ascending=False)
            .reset_index(drop=True)
        )

        ranking["posicao"] = ranking.index + 1

        return ranking

In [2]:
from validador import (
    validar_modelo,
    validar_modelo_por_segmento,
    gerar_relatorio,
    validar_modelo_por_tempo_relacionamento,
)


def separa_grupos(df_completo: pd.DataFrame):

    safras_validacao = [202509, 202510]
    safras_teste = [202511, 202512]

    df_treino = df_completo[~df_completo["safra"].isin(safras_teste + safras_validacao)]
    df_validacao = df_completo[df_completo["safra"].isin(safras_validacao)]
    df_teste = df_completo[df_completo["safra"].isin(safras_teste)]

    return df_treino, df_validacao, df_teste


df_clientes = pd.read_csv("data/clientes.csv")
df_interacoes = pd.read_csv("data/interacoes.csv")
df_contratos = pd.read_csv("data/contratos_ativos.csv")
df_produtos = pd.read_csv("data/produtos.csv")

baseline = Baseline(
    df_produtos["produto"].unique().tolist(), df_contratos, df_interacoes, df_clientes
)

df_treino, _, df_teste = separa_grupos(df_interacoes)
df_teste = pd.merge(
    df_teste,
    df_clientes[["segmento", "qtd_meses_cliente", "id_cliente"]],
    on=["id_cliente"],
)

relatorios = []

In [3]:
relatorio_aleatorio = gerar_relatorio(
    validar_modelo(df_teste, baseline.gerar_baseline_aleatorio),
    validar_modelo_por_segmento(df_teste, baseline.gerar_baseline_aleatorio),
    validar_modelo_por_tempo_relacionamento(
        df_teste, baseline.gerar_baseline_aleatorio
    ),
    "Aleatório",
)

relatorios.append(relatorio_aleatorio)
relatorio_aleatorio

Iniciando validação completa para 39906 casos de teste...


39906it [01:56, 341.33it/s]


Validando segmento: basico
Iniciando validação completa para 22067 casos de teste...


22067it [00:44, 496.12it/s]


Validando segmento: premium
Iniciando validação completa para 5908 casos de teste...


...

Validando clientes com meses de leracionamento: 3 a 6 meses
Iniciando validação completa para 3380 casos de teste...


3380it [00:03, 1080.66it/s]


Validando clientes com meses de leracionamento: 6 a 12 meses
Iniciando validação completa para 5361 casos de teste...


5361it [00:05, 940.73it/s] 


,Métricas Primárias.Precision@5,Métricas Primárias.NDCG@5 (Binário),Métricas Secundárias.Recall@5,Métricas Secundárias.Hit Rate@5,Métricas Secundárias.MAP@5,Métricas Secundárias.Catalog Coverage,Métricas de Receita.NDCG@5 (Receita),Métricas de Receita.Avg Revenue@5,Métricas de Receita.Revenue Recall@5,grupo,modelo
0,0.053782,0.153638,0.268908,0.268908,0.116293,1.0,0.153638,31.452101,0.268908,geral,Aleatório
1,0.051042,0.164902,0.255208,0.255208,0.135851,1.0,0.164902,29.348073,0.255208,basico,Aleatório
2,0.067500,0.170097,0.337500,0.337500,0.116458,1.0,0.170097,34.061000,0.337500,premium,Aleatório
3,0.077647,0.237078,0.388235,0.388235,0.188627,1.0,0.237078,36.089882,0.388235,intermediario,Aleatório
4,0.058065,0.167634,0.290323,0.290323,0.128495,1.0,0.167634,37.530968,0.290323,<3 meses,Aleatório
5,0.040000,0.122423,0.200000,0.200000,0.097500,1.0,0.122423,7.251000,0.200000,3 a 6 meses,Aleatório
6,0.045455,0.147956,0.227273,0.227273,0.121970,1.0,0.147956,23.166364,0.227273,6 a 12 meses,Aleatório


In [4]:
relatorio_popularidade = gerar_relatorio(
    validar_modelo(df_teste, baseline.gerar_baseline_popularidade),
    validar_modelo_por_segmento(df_teste, baseline.gerar_baseline_popularidade),
    validar_modelo_por_tempo_relacionamento(
        df_teste, baseline.gerar_baseline_popularidade
    ),
    "Popularidade",
)

relatorios.append(relatorio_popularidade)
relatorio_popularidade

Iniciando validação completa para 39906 casos de teste...


39906it [01:56, 341.19it/s]


Validando segmento: basico
Iniciando validação completa para 22067 casos de teste...


22067it [00:44, 492.16it/s]


Validando segmento: premium
Iniciando validação completa para 5908 casos de teste...


...

Validando clientes com meses de leracionamento: 3 a 6 meses
Iniciando validação completa para 3380 casos de teste...


3380it [00:03, 1069.99it/s]


Validando clientes com meses de leracionamento: 6 a 12 meses
Iniciando validação completa para 5361 casos de teste...


5361it [00:05, 939.94it/s] 


,Métricas Primárias.Precision@5,Métricas Primárias.NDCG@5 (Binário),Métricas Secundárias.Recall@5,Métricas Secundárias.Hit Rate@5,Métricas Secundárias.MAP@5,Métricas Secundárias.Catalog Coverage,Métricas de Receita.NDCG@5 (Receita),Métricas de Receita.Avg Revenue@5,Métricas de Receita.Revenue Recall@5,grupo,modelo
0,0.150140,0.536163,0.750700,0.750700,0.465079,0.45,0.536163,84.237143,0.750700,geral,Popularidade
1,0.164583,0.609870,0.822917,0.822917,0.538628,0.35,0.609870,103.202708,0.822917,basico,Popularidade
2,0.092500,0.245683,0.462500,0.462500,0.174792,0.45,0.245683,35.396500,0.462500,premium,Popularidade
3,0.171765,0.643066,0.858824,0.858824,0.572157,0.35,0.643066,87.364941,0.858824,intermediario,Popularidade
4,0.167742,0.585252,0.838710,0.838710,0.502688,0.35,0.585252,100.258065,0.838710,<3 meses,Popularidade
5,0.150000,0.460400,0.750000,0.750000,0.363333,0.40,0.460400,102.014500,0.750000,3 a 6 meses,Popularidade
6,0.150000,0.564938,0.750000,0.750000,0.502652,0.35,0.564938,77.538182,0.750000,6 a 12 meses,Popularidade


In [5]:
relatorio_popularidade_segmento = gerar_relatorio(
    validar_modelo(df_teste, baseline.gerar_baseline_popularidade_segmento),
    validar_modelo_por_segmento(
        df_teste, baseline.gerar_baseline_popularidade_segmento
    ),
    validar_modelo_por_tempo_relacionamento(
        df_teste, baseline.gerar_baseline_popularidade_segmento
    ),
    "Popularidade (segmento)",
)

relatorios.append(relatorio_popularidade_segmento)
relatorio_popularidade_segmento

Iniciando validação completa para 39906 casos de teste...


39906it [01:56, 341.29it/s]


Validando segmento: basico
Iniciando validação completa para 22067 casos de teste...


22067it [00:45, 486.32it/s]


Validando segmento: premium
Iniciando validação completa para 5908 casos de teste...


...

Validando clientes com meses de leracionamento: 3 a 6 meses
Iniciando validação completa para 3380 casos de teste...


3380it [00:03, 1036.37it/s]


Validando clientes com meses de leracionamento: 6 a 12 meses
Iniciando validação completa para 5361 casos de teste...


5361it [00:05, 911.57it/s]


,Métricas Primárias.Precision@5,Métricas Primárias.NDCG@5 (Binário),Métricas Secundárias.Recall@5,Métricas Secundárias.Hit Rate@5,Métricas Secundárias.MAP@5,Métricas Secundárias.Catalog Coverage,Métricas de Receita.NDCG@5 (Receita),Métricas de Receita.Avg Revenue@5,Métricas de Receita.Revenue Recall@5,grupo,modelo
0,0.186555,0.687866,0.932773,0.932773,0.605882,0.65,0.687866,101.750952,0.932773,geral,Popularidade (segmento)
1,0.194792,0.717948,0.973958,0.973958,0.632118,0.35,0.717948,106.281563,0.973958,basico,Popularidade (segmento)
2,0.192500,0.681199,0.962500,0.962500,0.587083,0.45,0.681199,107.661125,0.962500,premium,Popularidade (segmento)
3,0.162353,0.626192,0.811765,0.811765,0.564314,0.40,0.626192,85.954588,0.811765,intermediario,Popularidade (segmento)
4,0.180645,0.637747,0.903226,0.903226,0.548387,0.50,0.637747,101.908387,0.903226,<3 meses,Popularidade (segmento)
5,0.190000,0.765491,0.950000,0.950000,0.705833,0.65,0.765491,119.738000,0.950000,3 a 6 meses,Popularidade (segmento)
6,0.177273,0.685211,0.886364,0.886364,0.618182,0.65,0.685211,83.542273,0.886364,6 a 12 meses,Popularidade (segmento)


In [7]:
df_relatorios = pd.concat(relatorios).reset_index(drop=True)
df_relatorios.to_excel("relatorios/resultados_baseline.xlsx")
df_relatorios

,Métricas Primárias.Precision@5,Métricas Primárias.NDCG@5 (Binário),Métricas Secundárias.Recall@5,Métricas Secundárias.Hit Rate@5,Métricas Secundárias.MAP@5,Métricas Secundárias.Catalog Coverage,Métricas de Receita.NDCG@5 (Receita),Métricas de Receita.Avg Revenue@5,Métricas de Receita.Revenue Recall@5,grupo,modelo
0,0.053782,0.153638,0.268908,0.268908,0.116293,1.00,0.153638,31.452101,0.268908,geral,Aleatório
1,0.051042,0.164902,0.255208,0.255208,0.135851,1.00,0.164902,29.348073,0.255208,basico,Aleatório
2,0.067500,0.170097,0.337500,0.337500,0.116458,1.00,0.170097,34.061000,0.337500,premium,Aleatório
3,0.077647,0.237078,0.388235,0.388235,0.188627,1.00,0.237078,36.089882,0.388235,intermediario,Aleatório
4,0.058065,0.167634,0.290323,0.290323,0.128495,1.00,0.167634,37.530968,0.290323,<3 meses,Aleatório
5,0.040000,0.122423,0.200000,0.200000,0.097500,1.00,0.122423,7.251000,0.200000,3 a 6 meses,Aleatório
6,0.045455,0.147956,0.227273,0.227273,0.121970,1.00,0.147956,23.166364,0.227273,6 a 12 meses,Aleatório
7,0.150140,0.536163,0.750700,0.750700,0.465079,0.45,0.536163,84.237143,0.750700,geral,Popularidade
8,0.164583,0.609870,0.822917,0.822917,0.538628,0.35,0.609870,103.202708,0.822917,basico,Popularidade
9,0.092500,0.245683,0.462500,0.462500,0.174792,0.45,0.245683,35.396500,0.462500,premium,Popularidade
